# H&M Fashion Recommender System avec LightFM

---

## 📋 Projet EMIASD - Systèmes de Recommandation

**Auteur:** Malik Chettih  
**Date:** Octobre 2025  
**Dataset:** H&M Fashion (1.3M customers, 105K articles, 31M transactions)  
**Framework:** LightFM (Hybrid Recommender System)  

---

## 🎯 Objectifs du Projet

1. **Explorer** le dataset H&M Fashion et comprendre sa structure
2. **Échantillonner** intelligemment pour créer un dataset exploitable
3. **Construire** un système de recommandation hybride avec LightFM
4. **Comparer** différentes stratégies de train/test split
5. **Optimiser** les hyperparamètres du modèle
6. **Évaluer** les performances avec des métriques appropriées
7. **Analyser** les résultats et proposer des améliorations

---

## 🗂️ Structure du Notebook

**Section 0:** Configuration Globale et Imports  
**Section 1:** Exploration des Données (EDA)  
**Section 2:** Stratégie de Sampling  
**Section 3:** Prétraitement et Construction LightFM  
**Section 4:** Train/Test Split (3 Stratégies)  
**Section 5:** Entraînement des Modèles Baseline  
**Section 6:** Optimisation des Hyperparamètres  
**Section 7:** Évaluation Complète  
**Section 8:** Modèle Hybride et Analyse  
**Section 9:** Conclusions et Recommandations  

---

## ⚙️ Configuration Requise

```bash
pip install lightfm pandas numpy matplotlib seaborn scipy scikit-learn
```

**Temps d'exécution estimé:** ~15-20 minutes (avec SAMPLE_SIZE=50K)

---

---

# Section 0: Configuration Globale et Imports

---

## 0.1 Imports des Bibliothèques

In [1]:
# Imports standards
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
import json
import os
import time
from collections import Counter

# Imports scipy pour matrices sparse
from scipy.sparse import csr_matrix, coo_matrix
from sklearn.preprocessing import LabelEncoder

# Imports LightFM
try:
    from lightfm import LightFM
    from lightfm.data import Dataset
    from lightfm.evaluation import precision_at_k, recall_at_k, auc_score
    from lightfm.cross_validation import random_train_test_split
    print("✅ LightFM installé et disponible")
    LIGHTFM_AVAILABLE = True
except ImportError:
    print("❌ LightFM n'est pas installé!")
    print("   Installer avec: pip install lightfm")
    LIGHTFM_AVAILABLE = False
    raise ImportError("LightFM est requis pour ce notebook")

# Configuration des warnings et affichage
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 1000)

# Style des visualisations
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("\n✅ Imports terminés")

✅ LightFM installé et disponible

✅ Imports terminés


/opt/anaconda3/envs/lightfm_env/lib/python3.10/site-packages/lightfm/_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


## 0.2 Configuration Globale du Projet

In [2]:
# ============================================================================
# PARAMÈTRES GLOBAUX - À MODIFIER SELON VOS BESOINS
# ============================================================================

# Taille du sample (nombre de transactions à échantillonner)
SAMPLE_SIZE = 50000  # Options: 10000, 50000, 100000, 500000

# Stratégie de sampling
MIN_USER_TRANSACTIONS = 5   # Users avec au moins N transactions (dans dataset complet)
MIN_ITEM_TRANSACTIONS = 10  # Items avec au moins N transactions (dans dataset complet)

# Features sélectionnées
ITEM_FEATURE_COLUMNS = [
    'product_group_name',
    'index_group_name',
    'garment_group_name',
    'colour_group_name'
]

USER_FEATURE_COLUMNS = [
    'age_group',
    'club_member_status',
    'fashion_news_frequency'
]

# Paramètres de split
TEMPORAL_TRAIN_RATIO = 0.9  # 90% train, 10% test pour split temporel
RANDOM_TEST_PERCENTAGE = 0.2  # 20% test pour split aléatoire
USERBASED_TRAIN_RATIO = 0.8  # 80% users train, 20% users test

# Choix de la stratégie de split pour l'entraînement final
SPLIT_STRATEGY = 'random'  # Options: 'temporal', 'random', 'userbased'

# Paramètres d'entraînement
N_EPOCHS = 30
N_THREADS = 4

# Paramètres de grid search
GRID_SEARCH_PARAMS = {
    'no_components': [10, 30, 50, 100],
    'learning_rate': [0.01, 0.05, 0.1],
    'item_alpha': [1e-6, 1e-5, 1e-4],
    'user_alpha': [1e-6, 1e-5, 1e-4]
}

# Métriques d'évaluation
K_VALUES = [5, 10, 20]  # Pour Precision@K, Recall@K

# Seed pour reproductibilité
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Chemins des données
DATA_PATH = 'data/'
TRANSACTIONS_FILE = DATA_PATH + 'transactions_train.csv'
ARTICLES_FILE = DATA_PATH + 'articles.csv'
CUSTOMERS_FILE = DATA_PATH + 'customers.csv'

# ============================================================================
# AFFICHAGE DE LA CONFIGURATION
# ============================================================================

print("="*80)
print("CONFIGURATION DU PROJET")
print("="*80)
print(f"\n📊 Dataset:")
print(f"   • Taille du sample: {SAMPLE_SIZE:,} transactions")
print(f"   • Min transactions/user (filtrage): {MIN_USER_TRANSACTIONS}")
print(f"   • Min transactions/item (filtrage): {MIN_ITEM_TRANSACTIONS}")

print(f"\n🎯 Features:")
print(f"   • Item features: {len(ITEM_FEATURE_COLUMNS)} colonnes")
print(f"   • User features: {len(USER_FEATURE_COLUMNS)} colonnes")

print(f"\n🔀 Split Strategy:")
print(f"   • Stratégie choisie: {SPLIT_STRATEGY.upper()}")
print(f"   • Temporal ratio: {TEMPORAL_TRAIN_RATIO*100:.0f}% train")
print(f"   • Random test: {RANDOM_TEST_PERCENTAGE*100:.0f}%")

print(f"\n🤖 Entraînement:")
print(f"   • Epochs: {N_EPOCHS}")
print(f"   • Threads: {N_THREADS}")
print(f"   • Random state: {RANDOM_STATE}")

print(f"\n🔍 Grid Search:")
print(f"   • Paramètres à tester:")
for param, values in GRID_SEARCH_PARAMS.items():
    print(f"     - {param}: {values}")

total_combinations = np.prod([len(v) for v in GRID_SEARCH_PARAMS.values()])
print(f"   • Total combinaisons: {total_combinations}")

print(f"\n✅ Configuration chargée")
print("="*80)

CONFIGURATION DU PROJET

📊 Dataset:
   • Taille du sample: 50,000 transactions
   • Min transactions/user (filtrage): 5
   • Min transactions/item (filtrage): 10

🎯 Features:
   • Item features: 4 colonnes
   • User features: 3 colonnes

🔀 Split Strategy:
   • Stratégie choisie: RANDOM
   • Temporal ratio: 90% train
   • Random test: 20%

🤖 Entraînement:
   • Epochs: 30
   • Threads: 4
   • Random state: 42

🔍 Grid Search:
   • Paramètres à tester:
     - no_components: [10, 30, 50, 100]
     - learning_rate: [0.01, 0.05, 0.1]
     - item_alpha: [1e-06, 1e-05, 0.0001]
     - user_alpha: [1e-06, 1e-05, 0.0001]
   • Total combinaisons: 108

✅ Configuration chargée


## 0.3 Fonctions Utilitaires

In [3]:
def print_section_header(title, section_number=None):
    """Affiche un header de section formaté."""
    print("\n" + "="*80)
    if section_number:
        print(f"SECTION {section_number}: {title.upper()}")
    else:
        print(title.upper())
    print("="*80 + "\n")

def print_subsection_header(title):
    """Affiche un header de sous-section formaté."""
    print("\n" + "-"*80)
    print(title)
    print("-"*80 + "\n")

def print_dataframe_info(df, name):
    """Affiche des informations sur un DataFrame."""
    print(f"\n📊 {name}:")
    print(f"   • Shape: {df.shape}")
    print(f"   • Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"   • Colonnes: {list(df.columns)}")

def calculate_sparsity(n_interactions, n_users, n_items):
    """Calcule la sparsité d'une matrice user-item."""
    return 1 - (n_interactions / (n_users * n_items))

def format_large_number(num):
    """Formate un grand nombre avec des séparateurs."""
    return f"{num:,}"

def timer(func):
    """Décorateur pour mesurer le temps d'exécution."""
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"\n⏱️  Temps d'exécution: {end-start:.2f} secondes")
        return result
    return wrapper

def evaluate_model(model, test_interactions, train_interactions=None, 
                   item_features=None, user_features=None, k=10):
    """Évalue un modèle LightFM avec plusieurs métriques."""
    metrics = {}
    
    # Precision@K
    precision = precision_at_k(
        model, test_interactions, 
        train_interactions=train_interactions,
        item_features=item_features,
        user_features=user_features,
        k=k
    ).mean()
    metrics[f'precision@{k}'] = precision
    
    # Recall@K
    recall = recall_at_k(
        model, test_interactions,
        train_interactions=train_interactions,
        item_features=item_features,
        user_features=user_features,
        k=k
    ).mean()
    metrics[f'recall@{k}'] = recall
    
    # AUC
    auc = auc_score(
        model, test_interactions,
        train_interactions=train_interactions,
        item_features=item_features,
        user_features=user_features
    ).mean()
    metrics['auc'] = auc
    
    return metrics

print("✅ Fonctions utilitaires chargées")

✅ Fonctions utilitaires chargées
